# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

In [ ]:
!nvidia-smi

Mon Apr 27 21:16:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import torch

print("torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())
else:
    print("没有检测到 GPU，请重新设置 Runtime 为 GPU。")

torch version: 2.10.0+cu128
CUDA available: True
GPU name: Tesla T4
GPU count: 1


In [ ]:
!pip install -q -U "transformers>=4.51.0" accelerate bitsandbytes sentencepiece pandas numpy tqdm sympy
!pip install -q antlr4-python3-runtime==4.11.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 103.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 95.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 100.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.2 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but

In [ ]:
import torch
import transformers
import pandas as pd
import numpy as np
import sympy

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("sympy:", sympy.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

torch: 2.10.0+cu128
transformers: 5.6.2
pandas: 3.0.2
numpy: 2.0.2
sympy: 1.14.0
CUDA available: True
GPU: Tesla T4


In [ ]:
!pip install -q -U "pandas==2.2.2" "numpy==2.0.2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 120.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 97.2 MB/s eta 0:00:00


In [ ]:
import torch
import transformers
import pandas as pd
import numpy as np
import sympy

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("sympy:", sympy.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

torch: 2.10.0+cu128
transformers: 5.6.2
pandas: 3.0.2
numpy: 2.0.2
sympy: 1.14.0
CUDA available: True
GPU: Tesla T4


In [4]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import os

PROJECT_DIR = "/content/drive/MyDrive/151B_SP26_Competition"

print("PROJECT_DIR exists:", os.path.exists(PROJECT_DIR))
print("PROJECT_DIR:", PROJECT_DIR)

if os.path.exists(PROJECT_DIR):
    os.chdir(PROJECT_DIR)
    print("Current working directory:", os.getcwd())
    print("Files:")
    print(os.listdir(PROJECT_DIR))
else:
    print("没有找到 PROJECT_DIR，请检查路径。")

PROJECT_DIR exists: True
PROJECT_DIR: /content/drive/MyDrive/151B_SP26_Competition
Current working directory: /content/drive/MyDrive/151B_SP26_Competition
Files:
['utils.py', 'LICENSE', 'README.md', 'judger.py', 'data', '__pycache__', 'starter_code_cse151b_comp.ipynb', '.venv', 'results']


In [6]:
import os

DATA_DIR = os.path.join(PROJECT_DIR, "data")

print("DATA_DIR exists:", os.path.exists(DATA_DIR))
print("DATA_DIR:", DATA_DIR)

if os.path.exists(DATA_DIR):
    for name in os.listdir(DATA_DIR):
        path = os.path.join(DATA_DIR, name)
        size = os.path.getsize(path)
        print(f"{name:30s} {size} bytes")

DATA_DIR exists: True
DATA_DIR: /content/drive/MyDrive/151B_SP26_Competition/data
private.jsonl                  502352 bytes
public.jsonl                   668383 bytes


In [7]:
PUBLIC_PATH = os.path.join(PROJECT_DIR, "data", "public.jsonl")
PRIVATE_PATH = os.path.join(PROJECT_DIR, "data", "private.jsonl")
JUDGER_PATH = os.path.join(PROJECT_DIR, "judger.py")
UTILS_PATH = os.path.join(PROJECT_DIR, "utils.py")

print("PUBLIC_PATH:", PUBLIC_PATH)
print("exists:", os.path.exists(PUBLIC_PATH))

print("\nPRIVATE_PATH:", PRIVATE_PATH)
print("exists:", os.path.exists(PRIVATE_PATH))

print("\nJUDGER_PATH:", JUDGER_PATH)
print("exists:", os.path.exists(JUDGER_PATH))

print("\nUTILS_PATH:", UTILS_PATH)
print("exists:", os.path.exists(UTILS_PATH))


PUBLIC_PATH: /content/drive/MyDrive/151B_SP26_Competition/data/public.jsonl
exists: True

PRIVATE_PATH: /content/drive/MyDrive/151B_SP26_Competition/data/private.jsonl
exists: True

JUDGER_PATH: /content/drive/MyDrive/151B_SP26_Competition/judger.py
exists: True

UTILS_PATH: /content/drive/MyDrive/151B_SP26_Competition/utils.py
exists: True


In [8]:
import json

assert os.path.exists(PUBLIC_PATH), f"找不到 public.jsonl: {PUBLIC_PATH}"

def load_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line_idx, line in enumerate(f):
            line = line.strip()
            if line:
                try:
                    data.append(json.loads(line))
                except Exception as e:
                    print("JSON parse error at line:", line_idx)
                    print("line preview:", line[:500])
                    raise e
    return data

public_data = load_jsonl(PUBLIC_PATH)

print("public size:", len(public_data))
print("first item type:", type(public_data[0]))
print("first item keys:", public_data[0].keys())

print("\nFirst item preview:")
print(json.dumps(public_data[0], indent=2, ensure_ascii=False)[:2000])

public size: 1126
first item type: <class 'dict'>
first item keys: dict_keys(['question', 'answer', 'id'])

First item preview:
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


In [9]:
if os.path.exists(PRIVATE_PATH):
    private_data = load_jsonl(PRIVATE_PATH)

    print("private size:", len(private_data))
    print("first private item keys:", private_data[0].keys())

    print("\nFirst private item preview:")
    print(json.dumps(private_data[0], indent=2, ensure_ascii=False)[:2000])
else:
    private_data = None
    print("private.jsonl 不存在，先只处理 public。")

private size: 943
first private item keys: dict_keys(['question', 'id'])

First private item preview:
{
  "question": "Use the order of operations to simplify: a) $[13-(11-11)]-[8-(5-6)]=$ [ANS]\nb) $4 \\cdot 3-2+2 \\cdot 3=$ [ANS]",
  "id": 0
}


In [ ]:
import sys
import os

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print("sys.path[0]:", sys.path[0])

from judger import Judger

judger = Judger(strict_extract=False)

print("Judger loaded successfully.")

sys.path[0]: /content/drive/MyDrive/151B_SP26_Competition
Judger loaded successfully.


In [ ]:
import json
import os
import re

NOTEBOOK_PATH = os.path.join(PROJECT_DIR, "starter_code_cse151b_comp.ipynb")

with open(NOTEBOOK_PATH, "r", encoding="utf-8") as f:
    nb = json.load(f)

code_cells = []
for cell in nb["cells"]:
    if cell.get("cell_type") == "code":
        source = "".join(cell.get("source", []))
        if "Judger" in source or "auto_judge" in source or "public.jsonl" in source or "private.jsonl" in source:
            code_cells.append(source)

print("Relevant code cells:", len(code_cells))

for i, src in enumerate(code_cells):
    print("=" * 100)
    print("CELL", i)
    print(src[:3000])

Relevant code cells: 6
CELL 0
PUBLIC_PATH = os.path.join(PROJECT_DIR, "data", "public.jsonl")
PRIVATE_PATH = os.path.join(PROJECT_DIR, "data", "private.jsonl")
JUDGER_PATH = os.path.join(PROJECT_DIR, "judger.py")
UTILS_PATH = os.path.join(PROJECT_DIR, "utils.py")

print("PUBLIC_PATH:", PUBLIC_PATH)
print("exists:", os.path.exists(PUBLIC_PATH))

print("\nPRIVATE_PATH:", PRIVATE_PATH)
print("exists:", os.path.exists(PRIVATE_PATH))

print("\nJUDGER_PATH:", JUDGER_PATH)
print("exists:", os.path.exists(JUDGER_PATH))

print("\nUTILS_PATH:", UTILS_PATH)
print("exists:", os.path.exists(UTILS_PATH))

CELL 1
import json

assert os.path.exists(PUBLIC_PATH), f"找不到 public.jsonl: {PUBLIC_PATH}"

def load_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line_idx, line in enumerate(f):
            line = line.strip()
            if line:
                try:
                    data.append(json.loads(line))
                except Exception as e:
                 

In [ ]:
test_pred = r"The sum is \boxed{105950}."
test_gold = public_data[0]["answer"]

print("gold:", test_gold)
print("extracted:", judger.extract_ans(test_pred))

ok = judger.auto_judge(
    pred=test_pred,
    gold=test_gold,
    options=[[]] * len(test_gold),
)

print("judger result:", ok)

gold: ['325*(1+325)']
extracted: 105950
judger result: True


### Comment Out the cell below after first installation.

In [ ]:
# Install uv
!wget -qO- https://astral.sh/uv/install.sh | sh

# Create a virtual environment
!uv venv .venv --seed

# Install dependencies — this is fast thanks to uv's parallel resolver
!.venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# Install Jupyter Kernel
!.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

print("Done. Restart the kernel before proceeding.")
print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

downloading uv 0.11.8 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment with seed packages at: .venv
 + pip==26.1
Activate with: source .venv/bin/activate
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 22.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 13.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 46.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 646.8/646.8 kB 36.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 77.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 80.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.4/244.4 MB 71.8 MB/s  0:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 74.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 MB 69.8 MB/s  0:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

### Run the cell below every time to activate the installed environment.

In [ ]:
# activate venv after installation. This needs to be run everytime.
!source ./.venv/bin/activate

In [4]:
# 重新装包
!pip install -q sympy numpy tqdm bitsandbytes antlr4-python3-runtime==4.11.1
!pip install -q transformers vllm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 38.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
omegaconf 2.3.0 requires antlr4-python3-runtime==4.9.*, but you have antlr4-python3-runtime 4.11.1 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.4/244.4 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 111.5 MB/s et

In [ ]:
# Cell 1: 干净安装一个稳定组合
!pip uninstall -y vllm
!pip install vllm==0.10.2
!pip install -U nvidia-ml-py
!pip install -q sympy tqdm bitsandbytes antlr4-python3-runtime==4.11.1

In [9]:
# Cell: 卸载并安装稳定版
!pip uninstall -y vllm
!pip install vllm==0.10.2
!pip install -U nvidia-ml-py

  Using cached vllm-0.10.2-cp38-abi3-manylinux1_x86_64.whl.metadata (16 kB)
  Using cached llguidance-0.7.30-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (10 kB)
  Using cached outlines_core-0.2.11-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.8 kB)
  Using cached xgrammar-0.1.23-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.4 kB)
  Using cached setuptools-79.0.1-py3-none-any.whl.metadata (6.5 kB)
  Using cached compressed_tensors-0.11.0-py3-none-any.whl.metadata (7.0 kB)
  Using cached depyf-0.19.0-py3-none-any.whl.metadata (7.3 kB)
  Using cached numba-0.61.2-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (2.8 kB)
  Using cached ray-2.55.1-cp312-cp312-manylinux2014_x86_64.whl.metadata (21 kB)
  Using cached torch-2.8.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (30 kB)
  Using cached torchaudio-2.8.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (7.2 kB)
  Using cached torchvision-0.23.0-

In [7]:
!nvidia-smi
import torch, vllm
print("torch:", torch.__version__)         # 应该是 2.8.0
print("vllm:", vllm.__version__)            # 应该是 0.10.2
print("cuda available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0))

Sat May  2 18:17:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   34C    P0             53W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

ModuleNotFoundError: No module named 'vllm'

In [6]:
# Cell: 修复版本
!pip install -q transformers==4.56.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 79.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.3 MB/s eta 0:00:00


## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [3]:
import json
import os

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
DATA_PATH   = "data/private.jsonl"
OUTPUT_PATH = "results/samplepara-result.csv"
MAX_TOKENS  = 32768

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

ModuleNotFoundError: No module named 'vllm'

## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [10]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 943 questions  (300 MCQ, 643 free-form)

── MCQ sample ──
{
  "question": "Assuming the weights corresponding to the sign values are reduced by 1/10, then the arithmetic mean is ().",
  "options": [
    "Unchanged",
    "Increased by ten percent",
    "Reduced by one percent",
    "Increased by one percent",
    "Decreased by ten percent",
    "Halved",
    "Unable to determine",
    "Doubled",
    "Decreased by five percent",
    "Expanded tenfold"
  ],
  "id": 1
}

── Free-form sample ──
{
  "question": "Use the order of operations to simplify: a) $[13-(11-11)]-[8-(5-6)]=$ [ANS]\nb) $4 \\cdot 3-2+2 \\cdot 3=$ [ANS]",
  "id": 0
}


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [11]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

── MCQ user prompt (first 200 chars) ──
Assuming the weights corresponding to the sign values are reduced by 1/10, then the arithmetic mean is ().

Options:
A. Unchanged
B. Increased by ten percent
C. Reduced by one percent
D. Increased by  ...

── Free-form user prompt (first 200 chars) ──
Use the order of operations to simplify: a) $[13-(11-11)]-[8-(5-6)]=$ [ANS]
b) $4 \cdot 3-2+2 \cdot 3=$ [ANS] ...



In [ ]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem carefully but concisely. "
    "The problem may contain one or multiple blanks marked [ANS]. "
    "Return the final answer values in the same order as the blanks. "
    "If there are multiple answers, separate them by commas inside one single \\boxed{}. "
    "Do not include labels like a) or b) inside the box. "
    "End your response with exactly one final answer in the form \\boxed{...}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output only the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

## 5. Load Model with vLLM (for general case, vLLM is faster)

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_prefix_caching=False,
    gpu_memory_utilization=0.50,
    max_model_len=16384,
    trust_remote_code=True,
    max_num_seqs=256,
    max_num_batched_tokens=32768,
)

sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

print("Model loaded.")

In [12]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_prefix_caching=False,
    gpu_memory_utilization=0.85,
    max_model_len=16384,
    trust_remote_code=True,
    max_num_seqs=256,
    max_num_batched_tokens=32768,
)

sampling_params = SamplingParams(
    n=3,
    max_tokens=MAX_TOKENS,
    temperature=0.7,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.5,
    repetition_penalty=1.0,
)

print("Model loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

INFO 05-02 06:10:40 [__init__.py:742] Resolved architecture: Qwen3ForCausalLM


`torch_dtype` is deprecated! Use `dtype` instead!


INFO 05-02 06:10:40 [__init__.py:1815] Using max model len 16384
WARNING 05-02 06:10:40 [__init__.py:1217] bitsandbytes quantization is not fully optimized yet. The speed can be slower than non-quantized models.
INFO 05-02 06:10:44 [scheduler.py:222] Chunked prefill is enabled with max_num_batched_tokens=32768.
WARNING 05-02 06:10:45 [_ipex_ops.py:16] Import error msg: No module named 'intel_extension_for_pytorch'


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

WARNING 05-02 06:10:47 [__init__.py:2974] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
INFO 05-02 06:14:27 [llm.py:295] Supported_tasks: ['generate']
INFO 05-02 06:14:27 [__init__.py:36] No IOProcessor plugins requested by the model
Model loaded.


## 5. Load Model with Transformers (alternative to vLLM for DataHub)

We load **Qwen3-4B-Thinking-2507** with **INT4 quantization** via BitsAndBytes.  

Key parameters:
- `load_in_4bit` — quantization strategy of INT4

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

llm = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="auto",
)


## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

### Generate with vLLM

In [13]:
# Build prompts for first 5 entries
prompts = []
for item in data[:]:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

# Generate
print(f"Generating responses for {len(prompts)} questions...")
outputs = llm.generate(prompts, sampling_params=sampling_params)

responses = [out.outputs[0].text.strip() for out in outputs]

# Preview first 3
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

Generating responses for 943 questions...


Adding requests:   0%|          | 0/943 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/2829 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…


── Response 0 (id=0) ──
Okay, let's tackle these two problems one by one. I need to remember the order of operations, which is PEMDAS: Parentheses, Exponents, Multiplication and Division (left to right), Addition and Subtraction (left to right). Let me start with part a.

Part a: [13 - (11 - 11)] - [8 - (5 - 6)] = ?

First, I need to handle the innermost parentheses first. Let's look at the first bracket: [13 - (11 - 11) ...

── Response 1 (id=1) ──
Okay, let's try to figure out this problem. Hmm, the question says: "Assuming the weights corresponding to the sign values are reduced by 1/10, then the arithmetic mean is ()". Wait, the problem statement seems a bit incomplete or maybe there's some context missing here? Because it mentions "sign values" and weights. Hmm.

Wait, maybe this is related to weighted averages? Like, if you have a weigh ...

── Response 2 (id=2) ──
Okay, let's try to work through this problem step by step. First, I need to remember what the problem is asking. We

### Generate with Transformers (for Datahub)

In [ ]:
# Build prompts for first 5 entries
prompts = []
for item in data[:5]:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

# Tokenize (padded batch)
print(f"Generating responses for {len(prompts)} questions...")
inputs = tokenizer(
    prompts,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=16384,
).to(llm.device)

# Generate
with torch.no_grad():
    output_ids = llm.generate(
        **inputs,
        max_new_tokens=MAX_TOKENS,
        temperature=0.6,
        top_p=0.95,
        top_k=20,
        repetition_penalty=1.0,
        do_sample=True,
    )

# Decode only the new tokens (strip the prompt)
responses = []
for i, out in enumerate(output_ids):
    new_tokens = out[inputs["input_ids"].shape[1]:]
    responses.append(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())

# Preview first 3
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [ ]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

## 8. Summary

Print accuracy broken down by question type.

In [ ]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [ ]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

In [1]:
# 1. 构造 results
results = []
for item, response in zip(data, responses):
    results.append({
        "id":       item.get("id"),
        "is_mcq":   bool(item.get("options")),
        "response": response,
    })

# 2. 保存
SAVE_EVAL = False
out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, "w") as f:
    for r in results:
        record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

# 3. 验证一下
!wc -l {OUTPUT_PATH}
!head -1 {OUTPUT_PATH}

NameError: name 'data' is not defined

In [ ]:
import pandas as pd
import json

# 完整路径（注意文件夹名被截断了，需要展开看完整名字）
INPUT_PATH = "/content/drive/MyDrive/151B_SP26_Competition/results/submission-2.csv"
OUTPUT_PATH = "/content/drive/MyDrive/151B_SP26_Competition/results/submission.csv"

results = []
with open(INPUT_PATH) as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        results.append(json.loads(line))

# 只保留 id 和 response 两列
df = pd.DataFrame([
    {"id": r["id"], "response": r["response"]}
    for r in results
])

df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(df)} rows to {OUTPUT_PATH}")
print(df.head(2))

In [ ]:
df_check = pd.read_csv(OUTPUT_PATH)
print("行数:", len(df_check))
print("列名:", df_check.columns.tolist())
print(df_check.head(3))

In [2]:
# 检查 responses 是否还在
try:
    print(f"responses 还在，长度 = {len(responses)}")
    print(f"第一条预览: {responses[0][:100]}")
except NameError:
    print("responses 也没了")

# 检查之前的文件是否存在
import os
print("submission-2.csv 存在吗:", os.path.exists("/content/drive/MyDrive/151B_SP26_Competition/results/submission-2.csv"))

responses 也没了
submission-2.csv 存在吗: False


## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!